In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    BooleanType, LongType, MapType
)
from pyspark.sql.functions import window, col
from datetime import datetime

In [0]:
# project data catalog defined and created in <placeholder> notebook
catalog = 'wikimedia_db'

# db_schema containing unprocessed/streaming data
uc_schema_raw_events = 'raw_events'

# raw data is saved in a temp volume by yy_mm_day
raw_events_volume_time = datetime.now()
raw_events_volume =  f"events_tmp_{raw_events_volume_time.strftime('%y_%m_%d')}"
raw_data_path = f'/Volumes/{catalog}/{uc_schema_raw_events}/{raw_events_volume}'

# db schema for checkpointing streaming tables
db_schema_checkpoints = 'checkpoints'
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db_schema_checkpoints}")

DataFrame[]

In [0]:

# Meta schema (nested)
meta_schema = StructType([
    StructField("uri", StringType(), True),
    StructField("request_id", StringType(), True),
    StructField("id", StringType(), True),
    StructField("dt", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("stream", StringType(), True)
])

# Length schema (nested)
length_schema = StructType([
    StructField("old", IntegerType(), True),
    StructField("new", IntegerType(), True)
])

# Revision schema (nested)
revision_schema = StructType([
    StructField("old", LongType(), True),
    StructField("new", LongType(), True)
])

# Main recent change schema
recentchange_schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("meta", meta_schema, True),
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("length", length_schema, True),
    StructField("revision", revision_schema, True),
    StructField("server_url", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("parsedcomment", StringType(), True),
])


In [0]:
# Read data from a file
# Similar to definition of staticInputDF above, just using `readStream` instead of `read`
streamingInputDF = (
  spark
    .readStream                       
    .schema(recentchange_schema)               # Set the schema of the JSON data
    .option("maxFilesPerTrigger", 1)  # Treat a sequence of files as a stream by picking n number of files at a time
    .json(raw_data_path)
)


In [0]:
# Do some transformations
# Same query as staticInputDF
streamingCountsDF = (
  streamingInputDF
    .groupBy(
      streamingInputDF.bot, # group by edit made by bot boolean
      window(
        col("timestamp").cast("timestamp"), 
        "5 minutes"
      )
    )
    .count()
)


In [0]:
# temp volume for checkpoint storage
volume = 'tmp_streamingInputDF'
volume_path = f'/Volumes/{catalog}/{db_schema_checkpoints}/{volume}'
volume_name = f'{catalog}.{db_schema_checkpoints}.{volume}'

# drop old temp volume and recreate
spark.sql(f"DROP VOLUME IF EXISTS {volume_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

# Display the streaming dataframe
streamingInputDF.display(checkpointLocation=volume_path)

Checkpointing to /Volumes/wikimedia_db/checkpoints/tmp_streamingInputDF


In [0]:
# temp volume for checkpoint storage
volume = 'tmp_streamingDF'
volume_path = f'/Volumes/{catalog}/{db_schema_checkpoints}/{volume}'
volume_name = f'{catalog}.{db_schema_checkpoints}.{volume}'

# drop old temp volume and recreate
spark.sql(f"DROP VOLUME IF EXISTS {volume_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

# Display transformed data
streamingCountsDF.display(checkpointLocation=volume_path)

bot,window,count


In [0]:
from pyspark.sql.functions import count, col,current_timestamp, expr, to_timestamp, from_unixtime

In [0]:
#Sources that is from french wiki and for lionel messi
streamingFilteredmessi = streamingInputDF.filter(
    (col("wiki") == "frwiki") &
    (col("title") == "Lionel Messi")
)

In [0]:
#Sources where modifications has been made with a bot
streamingFilteredbot = streamingInputDF.filter(
    col("bot") == True
)

In [0]:
# Only on french page here
streamingFilteredFrenchPage = streamingInputDF.filter(
    col("wiki") == "frwiki"
)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS wikimedia.football_events;



In [0]:
%sql
CREATE VOLUME IF NOT EXISTS wikimedia.football_events.checkpoints;

In [0]:
checkpoint_path = "dbfs:/Volumes/wikimedia/football_events/checkpoints/frwiki_bronze"

#write the bronze table
(streamingFilteredFrenchPage.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(once=True) 
    .table("wikimedia.football_events.bronze_frwiki_events")
)


In [0]:

# load the Bronze table that contains all raw events
bronze = spark.table("wikimedia.football_events.bronze_frwiki_events")
# create a gold table that summarizes user contribution activity
gold_users = (
    bronze
    # Keep only events created by real users
    .filter(col("bot") == False)
    .groupBy("user")
    .agg(count("*").alias("edit_count"))
    .orderBy(col("edit_count").desc())
)
# Save the result as a Delta table in the Gold layer
gold_users.write.format("delta").mode("overwrite").saveAsTable(
    "wikimedia.football_events.gold_frwiki_user_activity"
)

display(spark.table("wikimedia.football_events.gold_frwiki_user_activity"))


user,edit_count
Simans9093,31
Monsieur de Vendôme,18
Thefan42,10
La Graillance,10
Baname LARE,8
Adri08,8
Gouttel'ocean,7
Arroser,7
Père Igor,7
Lestoille,7


In [0]:
display(
    spark.table("wikimedia.football_events.bronze_frwiki_events")
    .filter(col("user").like("%.%"))
)
spark.sql("CREATE SCHEMA IF NOT EXISTS wikimedia.football_events_alerts")
# Create a schema for alert tables if it does not exist yet
spark.sql("""
CREATE VOLUME IF NOT EXISTS wikimedia.football_events_alerts.rare_events_alerts
""")
bronze = spark.table("wikimedia.football_events.bronze_frwiki_events")
# filter events made by anonymous editors.the anonymous users represented by their IP addresses
rare_event_alerts = bronze.filter(
    col("user").rlike("^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+$") |  
    col("user").rlike("^[0-9a-fA-F:]+$")                    
)
# save the filtered anonymous activity into an alerts table
rare_event_alerts.write.mode("overwrite").format("delta").saveAsTable(
    "wikimedia.football_events_alerts.anonymous_edit_alerts"
)

display(spark.table("wikimedia.football_events_alerts.anonymous_edit_alerts"))


$schema,meta,id,type,namespace,title,comment,timestamp,user,bot,minor,patrolled,length,revision,server_url,server_name,wiki,parsedcomment
/mediawiki/recentchange/1.0.0,"List(https://fr.wikipedia.org/wiki/Le_Roi_des_rois_(film,_2025), cc21153f-1fd4-4f54-a678-76282cafb2b6, d6b9f996-67ca-47fb-ad79-76460c9d349d, 2025-11-14T15:13:14.329Z, fr.wikipedia.org, mediawiki.recentchange)",561112339,edit,0,"Le Roi des rois (film, 2025)",/* Synopsis */,1763133193,Lucy.Piano,false,false,false,"List(14672, 14712)","List(230639977, 230640129)",https://fr.wikipedia.org,fr.wikipedia.org,frwiki,→Synopsis
/mediawiki/recentchange/1.0.0,"List(https://fr.wikipedia.org/wiki/Utilisateur:Lucie.da1990/Brouillon, bdba8fd9-6046-48f0-b035-f87083672435, 158f677a-aab6-4ae4-95a5-871103cde1ba, 2025-11-14T15:26:31.666Z, fr.wikipedia.org, mediawiki.recentchange)",561112835,edit,2,Utilisateur:Lucie.da1990/Brouillon,Ajout du paragraphe sur l'histoire de Chabé (partie 3 et 4=,1763133990,Lucie.da1990,false,false,false,"List(1816, 4260)","List(229758126, 230640406)",https://fr.wikipedia.org,fr.wikipedia.org,frwiki,Ajout du paragraphe sur l'histoire de Chabé (partie 3 et 4=


$schema,meta,id,type,namespace,title,comment,timestamp,user,bot,minor,patrolled,length,revision,server_url,server_name,wiki,parsedcomment


In [0]:

bronze_table = "wikimedia.football_events.bronze_frwiki_events"
bronze = spark.table(bronze_table)


# Try converting both possibilities (ms and seconds).
bronze = bronze.withColumn(
    "event_ts",
    expr("CASE WHEN timestamp > 1000000000000 THEN to_timestamp(timestamp/1000) ELSE to_timestamp(timestamp) END")
)

# data from last hour
cleaned = bronze.filter(
    col("event_ts") >= expr("current_timestamp() - INTERVAL 1 DAY")
)

# Overwrite the bronze table with cleaned data
cleaned.drop("event_ts").write.mode("overwrite").format("delta").saveAsTable(bronze_table)
